# 🚀 PRISM AI — Qwen2.5-1.5B QLoRA Automated Fine-Tuning
### MoSPI Infrastructure Risk Intelligence Advisory LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant1506/SIH-26/blob/main/ml/notebooks/PRISM_QLoRA_Colab_FineTuning.ipynb)

This notebook provides **100% automated zero-touch fine-tuning** of `Qwen/Qwen2.5-1.5B-Instruct` using 4-bit NF4 Quantization and PEFT LoRA.

### ⚡ Quick Start:
1. Switch runtime to GPU: **`Runtime > Change runtime type > T4 GPU`**
2. Click **`Runtime > Run all`** (or press `Ctrl + F9`)
3. The script automatically fetches data, fine-tunes the model (~10-12 mins), tests inference, and saves the adapter!

---

## 1. Verify GPU Setup

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies

In [ ]:
!pip install -q transformers peft bitsandbytes trl datasets accelerate scipy

## 3. Automated Dataset Fetching
Automatically fetches `llm_train.jsonl` from GitHub repository, or prompts for upload if offline.

In [ ]:
import os
import urllib.request

DATASET_PATH = 'llm_train.jsonl'
RAW_URL = 'https://raw.githubusercontent.com/vedant1506/SIH-26/main/ml/data/processed/llm_train.jsonl'

if not os.path.exists(DATASET_PATH):
    print(f'Attempting automatic download from GitHub repository...')
    try:
        urllib.request.urlretrieve(RAW_URL, DATASET_PATH)
        print(f'✅ Successfully downloaded {DATASET_PATH} ({os.path.getsize(DATASET_PATH):,} bytes).')
    except Exception as e:
        print(f'GitHub fetch failed ({e}). Please upload llm_train.jsonl manually:')
        from google.colab import files
        files.upload()
else:
    print(f'✅ {DATASET_PATH} already present ({os.path.getsize(DATASET_PATH):,} bytes).')

## 4. Inspect Sample Training Data

In [ ]:
import json

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())

print('Sample ID:', sample.get('id'))
for msg in sample.get('messages', []):
    print(f"\n--- [{msg['role'].upper()}] ---\n{msg['content']}")

## 5. Load Base Model with 4-bit NF4 Quantization & PEFT LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = 'prism_qwen2.5_qlora_adapter'

print(f'Loading tokenizer: {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Configuring 4-bit NF4 Quantization...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading base LLM: {MODEL_ID}...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)
print('✅ Base Model & LoRA successfully configured.')

## 6. Execute Fine-Tuning with SFTTrainer

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset('json', data_files={'train': DATASET_PATH})
print(f'Total training records: {len(dataset["train"])}')

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=1024,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    peft_config=peft_config,
    args=sft_config,
)

print('🔥 Running QLoRA Fine-Tuning on GPU...')
trainer.train()
print('🎉 Fine-Tuning Complete!')


## 7. Verify Inference with Fine-Tuned LoRA Adapter

In [ ]:
# Save trained adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved adapter weights to {OUTPUT_DIR}')

# Test prompt
test_prompt = """Analyze infrastructure project #0999.
- Original Sanctioned Budget: ₹450.00 Cr
- Revised Estimate Budget: ₹620.00 Cr
- Cumulative Expenditure: ₹380.00 Cr
- Reported Physical Progress: 48.0%
- Burn Rate vs Progress Gap: +12.5%
- Schedule Elapsed Ratio: 78.0%
- Cost Overrun Status: Yes (+37.8%)
- Schedule Delay Status: Yes (14.2 months)"""

messages = [
    {"role": "system", "content": "You are PRISM AI, an advanced infrastructure risk intelligence system for major capital projects in India. Analyze the provided financial expenditure, physical progress, and schedule metrics to generate a precise executive risk narrative."},
    {"role": "user", "content": test_prompt}
]
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3, top_p=0.9)
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print('\n=== INFERENCE TEST OUTPUT ===\n', response)

## 8. Export Adapter: Download or Save to Google Drive

In [ ]:
import shutil
from google.colab import files

# 1. Compress into zip archive
zip_filename = 'prism_qwen2.5_qlora_adapter.zip'
shutil.make_archive('prism_qwen2.5_qlora_adapter', 'zip', OUTPUT_DIR)
print(f'✅ Compressed adapter to {zip_filename}.')

# 2. Optional: Save to Google Drive if mounted
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_dest = '/content/drive/MyDrive/prism_qwen2.5_qlora_adapter.zip'
    shutil.copyfile(zip_filename, drive_dest)
    print(f'✅ Successfully saved copy to Google Drive: {drive_dest}')
except Exception as e:
    print(f'Drive mount skipped ({e}). Triggering browser direct download...')

# 3. Trigger browser download
files.download(zip_filename)